In [ ]:
import torch
import plotly.graph_objects as go

from src.gp_ccm import run_sigGPCCM_experiment
from src.sp_ccm import run_ccm_experiment

torch.set_printoptions(sci_mode = False)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)
print()

In [7]:
# Python package versions used
%load_ext watermark
%watermark --python
%watermark --iversions

Python implementation: CPython
Python version       : 3.11.3
IPython version      : 8.18.0

plotly: 5.9.0
torch : 2.1.1+cu118



# Data generation

In [ ]:
# Set length of timeseries
N_length = torch.tensor([400])

# Initialise values at t = 0
c = torch.tensor([0.2])
a = torch.tensor([0.3])
b = torch.tensor([0.4])

# Autoregressive function
for t in range(N_length - 1):
    
    c_next = c[t] * (3.9 - (3.9 * c[t]))
    a_next = a[t] * (3.8 - (3.8 * a[t]) - (0.3 * c[t]))
    b_next = b[t] * (3.6 - (3.6 * b[t]) - (0.3 * c[t]))

    c = torch.concat((c, c_next.unsqueeze(0)))
    a = torch.concat((a, a_next.unsqueeze(0)))
    b = torch.concat((b, b_next.unsqueeze(0)))
            
# Normalising step
c_norm = c.sub(c.mean(dim = -1).unsqueeze(-1)).div(c.std(dim = -1).unsqueeze(-1))
a_norm = a.sub(a.mean(dim = -1).unsqueeze(-1)).div(a.std(dim = -1).unsqueeze(-1))
b_norm = b.sub(b.mean(dim = -1).unsqueeze(-1)).div(b.std(dim = -1).unsqueeze(-1))

In [ ]:
torch.corrcoef(torch.stack((c_norm, a_norm)))

In [ ]:
torch.corrcoef(torch.stack((c_norm, b_norm)))

In [ ]:
torch.corrcoef(torch.stack((a_norm, b_norm)))

# Visualise

In [ ]:
fig = go.Figure()

fig.add_trace(go.Scatter(x = torch.arange(0, c_norm.shape[0])[0:41], y = c[0:41],
                    mode = 'lines',
                    name = 'C',
                    line_color = "forestgreen"))

fig.add_trace(go.Scatter(x = torch.arange(0, c_norm.shape[0])[0:41], y = a[0:41],
                    mode = 'lines',
                    name = 'A',
                    line_color = "blue"))

fig.add_trace(go.Scatter(x = torch.arange(0, c_norm.shape[0])[0:41], y = b[0:41],
                    mode = 'lines',
                    name = 'B',
                    line_color = "cornflowerblue"))

fig.update_layout(title = 'Confounding time series',
                   xaxis_title = 't',
                   yaxis_title = 'values')

fig.update_layout(template = "plotly_white")
fig.update_layout(font_family = "Lato")
fig.update_layout(xaxis_range=[-2,41])

fig.update_layout(autosize = False, width = 1000, height = 400)

fig.show()

# Define Hyperparameters

In [ ]:
# GLOBALS
k = 3
N_TRAIN = torch.tensor([100]).to(device)

##############
### GP-CCM ###
##############

sig_filter = torch.ones(size = (k, )).to(device)
sig_shift = torch.tensor(sig_filter.shape[0] - 1).to(device) # k -1 

NOISE_SCALE = torch.tensor([0.05], device = device) # for diagonal
NOISE_SCALE_medium = torch.tensor([0.02], device = device) # for diagonal
NOISE_SCALE_low = torch.tensor([0.01], device = device) # for diagonal

RBF_SCALE = torch.tensor([0.3], device = device)

############
### ECCM ###
############

ccm_filter = torch.ones(size = (k, )).to(device) # same as sig filter
ccm_shift = torch.tensor(ccm_filter.shape[0] - 1).to(device)

# C -> A

In [ ]:
### sig-GP_CCM ###
CA_gpccm_rho_mean, CA_gpccm_rho_sd, CA_gpccm_rho_ind_p95 =  run_sigGPCCM_experiment(
    causal_x = c_norm.to(device),
    causal_y = a_norm.to(device),
    sig_filter = sig_filter.to(device),
    sig_shift = sig_shift.to(device),
    rbf_scale = RBF_SCALE, 
    noise_scale = NOISE_SCALE, 
    n_train = N_TRAIN.to(device),
    device = device)

In [ ]:
CA_sigccm_confounding_results = torch.stack((CA_gpccm_rho_mean, CA_gpccm_rho_sd, CA_gpccm_rho_ind_p95.squeeze()))
# torch.save(CA_sigccm_confounding_results, "results/confounding/CA_sigccm_confounding_results.pt")

In [ ]:
### CCM ###
CA_ccm_rho_mean, CA_ccm_rho_sd, CA_ccm_rho_ind_p95, CA_ccm_noise =  run_ccm_experiment(
    causal_x = c_norm.to(device),
    causal_y = a_norm.to(device),
    ccm_filter = ccm_filter.to(device),
    ccm_shift = ccm_shift.to(device),
    n_train = N_TRAIN.to(device),
    device = device)

In [ ]:
CA_ccm_confounding_results = torch.stack((CA_ccm_rho_mean, CA_ccm_rho_sd, CA_ccm_rho_ind_p95.squeeze(), CA_ccm_noise.squeeze()))
# torch.save(CA_ccm_confounding_results, "results/confounding/CA_ccm_confounding_results.pt")

# C -> B

In [ ]:
### sig-GP_CCM ###
CB_gpccm_rho_mean, CB_gpccm_rho_sd, CB_gpccm_rho_ind_p95 =  run_sigGPCCM_experiment(
    causal_x = c_norm.to(device),
    causal_y = b_norm.to(device),
    sig_filter = sig_filter.to(device),
    sig_shift = sig_shift.to(device),
    rbf_scale = RBF_SCALE, 
    noise_scale = NOISE_SCALE, 
    n_train = N_TRAIN.to(device),
    device = device)

In [ ]:
CB_sigccm_confounding_results = torch.stack((CB_gpccm_rho_mean, CB_gpccm_rho_sd, CB_gpccm_rho_ind_p95.squeeze()))
# torch.save(CB_sigccm_confounding_results, "results/confounding/CB_sigccm_confounding_results.pt")

In [ ]:
### CCM ###
CB_ccm_rho_mean, CB_ccm_rho_sd, CB_ccm_rho_ind_p95, CB_ccm_noise =  run_ccm_experiment(
    causal_x = c_norm.to(device),
    causal_y = b_norm.to(device),
    ccm_filter = ccm_filter.to(device),
    ccm_shift = ccm_shift.to(device),
    n_train = N_TRAIN.to(device),
    device = device)

In [ ]:
CB_ccm_confounding_results = torch.stack((CB_ccm_rho_mean, CB_ccm_rho_sd, CB_ccm_rho_ind_p95.squeeze(), CA_ccm_noise.squeeze()))
# torch.save(CB_ccm_confounding_results, "results/confounding/CB_ccm_confounding_results.pt")

# A -> B

In [ ]:
### sig-GP_CCM ###
AB_gpccm_rho_mean, AB_gpccm_rho_sd, AB_gpccm_rho_ind_p95 =  run_sigGPCCM_experiment(
    causal_x = a_norm.to(device),
    causal_y = b_norm.to(device),
    sig_filter = sig_filter.to(device),
    sig_shift = sig_shift.to(device),
    rbf_scale = RBF_SCALE, 
    noise_scale = NOISE_SCALE, 
    n_train = N_TRAIN.to(device),
    device = device)

In [ ]:
AB_sigccm_confounding_results = torch.stack((AB_gpccm_rho_mean, AB_gpccm_rho_sd, AB_gpccm_rho_ind_p95.squeeze()))
# torch.save(AB_sigccm_confounding_results, "results/confounding/AB_sigccm_confounding_results.pt")

In [ ]:
### CCM ###
AB_ccm_rho_mean, AB_ccm_rho_sd, AB_ccm_rho_ind_p95, AB_ccm_noise =  run_ccm_experiment(
    causal_x = a_norm.to(device),
    causal_y = b_norm.to(device),
    ccm_filter = ccm_filter.to(device),
    ccm_shift = ccm_shift.to(device),
    n_train = N_TRAIN.to(device),
    device = device)

In [ ]:
AB_ccm_confounding_results = torch.stack((AB_ccm_rho_mean, AB_ccm_rho_sd, AB_ccm_rho_ind_p95.squeeze(), AB_ccm_noise.squeeze()))
# torch.save(AB_ccm_confounding_results, "results/confounding/AB_ccm_confounding_results.pt")

# A -> C

In [ ]:
### sig-GP_CCM ###
AC_gpccm_rho_mean, AC_gpccm_rho_sd, AC_gpccm_rho_ind_p95 =  run_sigGPCCM_experiment(
    causal_x = a_norm.to(device),
    causal_y = c_norm.to(device),
    sig_filter = sig_filter.to(device),
    sig_shift = sig_shift.to(device),
    rbf_scale = RBF_SCALE, 
    noise_scale = NOISE_SCALE, 
    n_train = N_TRAIN.to(device),
    device = device)

In [ ]:
AC_sigccm_confounding_results = torch.stack((AC_gpccm_rho_mean, AC_gpccm_rho_sd, AC_gpccm_rho_ind_p95.squeeze()))
# torch.save(AC_sigccm_confounding_results, "results/confounding/AC_sigccm_confounding_results.pt")

In [ ]:
### CCM ###
AC_ccm_rho_mean, AC_ccm_rho_sd, AC_ccm_rho_ind_p95, AC_ccm_noise =  run_ccm_experiment(
    causal_x = a_norm.to(device),
    causal_y = c_norm.to(device),
    ccm_filter = ccm_filter.to(device),
    ccm_shift = ccm_shift.to(device),
    n_train = N_TRAIN.to(device),
    device = device)

In [ ]:
AC_ccm_confounding_results = torch.stack((AC_ccm_rho_mean, AC_ccm_rho_sd, AC_ccm_rho_ind_p95.squeeze(), AC_ccm_noise.squeeze()))
# torch.save(AC_ccm_confounding_results, "results/confounding/AC_ccm_confounding_results.pt")

# B -> A

In [ ]:
### sig-GP_CCM ###
BA_gpccm_rho_mean, BA_gpccm_rho_sd, BA_gpccm_rho_ind_p95 =  run_sigGPCCM_experiment(
    causal_x = b_norm.to(device),
    causal_y = a_norm.to(device),
    sig_filter = sig_filter.to(device),
    sig_shift = sig_shift.to(device),
    rbf_scale = RBF_SCALE, 
    noise_scale = NOISE_SCALE, 
    n_train = N_TRAIN.to(device),
    device = device)

In [ ]:
BA_sigccm_confounding_results = torch.stack((BA_gpccm_rho_mean, BA_gpccm_rho_sd, BA_gpccm_rho_ind_p95.squeeze()))
# torch.save(BA_sigccm_confounding_results, "results/confounding/BA_sigccm_confounding_results.pt")

In [ ]:
### CCM ###
BA_ccm_rho_mean, BA_ccm_rho_sd, BA_ccm_rho_ind_p95, BA_ccm_noise =  run_ccm_experiment(
    causal_x = b_norm.to(device),
    causal_y = a_norm.to(device),
    ccm_filter = ccm_filter.to(device),
    ccm_shift = ccm_shift.to(device),
    n_train = N_TRAIN.to(device),
    device = device)

In [ ]:
BA_ccm_confounding_results = torch.stack((BA_ccm_rho_mean, BA_ccm_rho_sd, BA_ccm_rho_ind_p95.squeeze(), BA_ccm_noise.squeeze()))
# torch.save(BA_ccm_confounding_results, "results/confounding/BA_ccm_confounding_results.pt")

# B -> C

In [ ]:
### sig-GP_CCM ###
BC_gpccm_rho_mean, BC_gpccm_rho_sd, BC_gpccm_rho_ind_p95 =  run_sigGPCCM_experiment(
    causal_x = b_norm.to(device),
    causal_y = c_norm.to(device),
    sig_filter = sig_filter.to(device),
    sig_shift = sig_shift.to(device),
    rbf_scale = RBF_SCALE, 
    noise_scale = NOISE_SCALE, 
    n_train = N_TRAIN.to(device),
    device = device)

In [ ]:
BC_sigccm_confounding_results = torch.stack((BC_gpccm_rho_mean, BC_gpccm_rho_sd, BC_gpccm_rho_ind_p95.squeeze()))
# torch.save(BC_sigccm_confounding_results, "results/confounding/BC_sigccm_confounding_results.pt")

In [ ]:
### CCM ###
BC_ccm_rho_mean, BC_ccm_rho_sd, BC_ccm_rho_ind_p95, BC_ccm_noise =  run_ccm_experiment(
    causal_x = b_norm.to(device),
    causal_y = c_norm.to(device),
    ccm_filter = ccm_filter.to(device),
    ccm_shift = ccm_shift.to(device),
    n_train = N_TRAIN.to(device),
    device = device)

In [ ]:
BC_ccm_confounding_results = torch.stack((BC_ccm_rho_mean, BC_ccm_rho_sd, BC_ccm_rho_ind_p95.squeeze(), BC_ccm_noise.squeeze()))
# torch.save(BC_ccm_confounding_results, "results/confounding/BC_ccm_confounding_results.pt")